# 02 · Ensembles (Voting, Bagging, Stacking)

Cell numbers count the cells after this note, starting at 0.

Notebook that produced the ensemble results in Table 2 of the manuscript: Voting and Bagging under SI, and Voting, Bagging and Stacking under SD. Stacking under SI is in notebook 01, cells 25-27.
Outputs are kept as saved; the data path was replaced with `/path/to/Kaggle Dataset`. A final cell that built a PDF summary report was removed because it drew illustrative values from random numbers and is not a result source.

Note on Stacking: the meta-learner is trained on base-model predictions for the same rows the base models were fitted on. See `docs/PROVENANCE.md`.


In [ ]:
# ============================================================
# COMPREHENSIVE ENSEMBLE LEARNING - ALL METHODS
# Queue execution: Run all cells sequentially
# Total time: ~6 hours
# ============================================================

import time
import os
import gc
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (AdaBoostClassifier, RandomForestClassifier,
                               VotingClassifier, BaggingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from catboost import CatBoostClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("COMPREHENSIVE ENSEMBLE LEARNING - ALL METHODS")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\nStructure:")
print("  - 3 Ensemble Methods: Voting, Bagging, Stacking")
print("  - 2 Validations: Subject-Dependent, Subject-Independent")
print("  - 3 Tasks: Relax-Stroop, Relax-Arithmetic, Relax-Mirror")
print("  - Total: 18 experiments (15 pending)")
print("="*80)

# Paths
BASE_DATA_PATH = "/path/to/Kaggle Dataset"
BASE_RESULTS_PATH = f"{BASE_DATA_PATH}/Results/ResultsAfterOptimization"
ENSEMBLE_PATH = f"{BASE_RESULTS_PATH}/Complete_Ensemble_Analysis"
os.makedirs(ENSEMBLE_PATH, exist_ok=True)

# Load functions
def load_data_stratified(folder_path, label):
    """For subject-dependent (no groups)"""
    files = sorted(glob.glob(f"{folder_path}/*.csv"))
    dfs = []
    for f in files:
        df = pd.read_csv(f)
        if df.shape[1] > 1: df = df.iloc[:, 1:]
        dfs.append(df)
    data = pd.concat(dfs, ignore_index=True)
    return data.values.astype(np.float32), np.full(len(data), label)

def load_data_grouped(folder_path, label):
    """For subject-independent (with groups)"""
    files = sorted(glob.glob(f"{folder_path}/*.csv"))
    dfs, groups = [], []
    for i, f in enumerate(files):
        df = pd.read_csv(f)
        if df.shape[1] > 1: df = df.iloc[:, 1:]
        dfs.append(df)
        groups.append(np.full(len(df), f"label{label}_sub{i}"))
    data = pd.concat(dfs, ignore_index=True)
    return data.values.astype(np.float32), np.full(len(data), label), np.hstack(groups)

# Base models
def get_base_models():
    return {
        'RandomForest': RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=4, random_state=42),
        'XGBoost': xgb.XGBClassifier(n_estimators=100, max_depth=6, n_jobs=2, random_state=42, eval_metric='logloss'),
        'AdaBoost': AdaBoostClassifier(n_estimators=100, learning_rate=0.8, random_state=42),
        'CatBoost': CatBoostClassifier(iterations=100, depth=6, random_state=42, verbose=False, thread_count=4)
    }

# Tasks
TASKS = [
    ("Relax", "Stroop", "Relax-Stroop"),
    ("Relax", "Arithmetic", "Relax-Arithmetic"),
    ("Relax", "Mirror_image", "Relax-Mirror")
]

print("\n✅ Setup complete. Ready for sequential execution.\n")

COMPREHENSIVE ENSEMBLE LEARNING - ALL METHODS
Started: 2026-01-30 13:20:50

Structure:
  - 3 Ensemble Methods: Voting, Bagging, Stacking
  - 2 Validations: Subject-Dependent, Subject-Independent
  - 3 Tasks: Relax-Stroop, Relax-Arithmetic, Relax-Mirror
  - Total: 18 experiments (15 pending)

✅ Setup complete. Ready for sequential execution.



In [ ]:
# ============================================================
# METHOD 1: VOTING ENSEMBLE - SUBJECT-INDEPENDENT
# Expected: ~1 hour (3 tasks × 20 min each)
# ============================================================

print("\n" + "="*80)
print("METHOD 1: VOTING ENSEMBLE - SUBJECT-INDEPENDENT")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

voting_si_results = []
method_start = time.time()

for task_idx, (d1, d2, task_name) in enumerate(TASKS, 1):
    print(f"\n{'='*80}")
    print(f"TASK {task_idx}/3: {task_name}")
    print(f"{'='*80}")

    # Load with groups
    X1, y1, g1 = load_data_grouped(f"{BASE_DATA_PATH}/{d1}", 0)
    X2, y2, g2 = load_data_grouped(f"{BASE_DATA_PATH}/{d2}", 1)
    X = np.vstack((X1, X2))
    y = np.hstack((y1, y2))
    groups = np.hstack((g1, g2))
    print(f"Loaded: {len(X):,} samples")

    gkf = GroupKFold(n_splits=5)
    fold_accs = []

    task_start = time.time()

    for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), 1):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Voting ensemble
        base_models = get_base_models()
        voting_clf = VotingClassifier(
            estimators=[(name, model) for name, model in base_models.items()],
            voting='soft',
            n_jobs=1
        )

        voting_clf.fit(X_train_scaled, y_train)
        y_pred = voting_clf.predict(X_test_scaled)
        acc = accuracy_score(y_test, y_pred)
        fold_accs.append(acc)

        print(f"  Fold {fold_idx}/5: {acc:.4f}")

    avg_acc = np.mean(fold_accs)
    std_acc = np.std(fold_accs)

    print(f"\n  Result: {avg_acc:.4f} ± {std_acc:.4f} ({avg_acc*100:.2f}%)")
    print(f"  Time: {(time.time()-task_start)/60:.1f} min")

    voting_si_results.append({
        'Task': task_name,
        'Accuracy': avg_acc,
        'Std': std_acc
    })

    del X, y, groups
    gc.collect()

# Save
df = pd.DataFrame(voting_si_results)
df.to_csv(f"{ENSEMBLE_PATH}/Voting_SubjectIndependent.csv", index=False)

print(f"\n{'='*80}")
print("VOTING SUBJECT-INDEPENDENT COMPLETE")
print(f"{'='*80}")
print(f"Average: {df['Accuracy'].mean():.4f}")
print(f"Total time: {(time.time()-method_start)/60:.1f} min")
print(f"Saved: {ENSEMBLE_PATH}/Voting_SubjectIndependent.csv\n")


METHOD 1: VOTING ENSEMBLE - SUBJECT-INDEPENDENT
Started: 2026-01-30 10:56:26


TASK 1/3: Relax-Stroop
Loaded: 768,000 samples
  Fold 1/5: 0.5628
  Fold 2/5: 0.5968
  Fold 3/5: 0.4830
  Fold 4/5: 0.5358
  Fold 5/5: 0.5153

  Result: 0.5387 ± 0.0390 (53.87%)
  Time: 32.8 min

TASK 2/3: Relax-Arithmetic
Loaded: 768,000 samples
  Fold 1/5: 0.4683
  Fold 2/5: 0.5405
  Fold 3/5: 0.4173
  Fold 4/5: 0.4564
  Fold 5/5: 0.4510

  Result: 0.4667 ± 0.0406 (46.67%)
  Time: 33.4 min

TASK 3/3: Relax-Mirror
Loaded: 768,000 samples
  Fold 1/5: 0.5300
  Fold 2/5: 0.5854
  Fold 3/5: 0.4236
  Fold 4/5: 0.4870
  Fold 5/5: 0.5091

  Result: 0.5070 ± 0.0530 (50.70%)
  Time: 33.3 min

VOTING SUBJECT-INDEPENDENT COMPLETE
Average: 0.5042
Total time: 99.6 min
Saved: /path/to/Kaggle Dataset/Results/ResultsAfterOptimization/Complete_Ensemble_Analysis/Voting_SubjectIndependent.csv



In [ ]:
# ============================================================
# METHOD 2: BAGGING - SUBJECT-INDEPENDENT (OPTIMIZED)
# ============================================================

try:
    print("\n" + "="*80)
    print("METHOD 2: BAGGING ENSEMBLE - SUBJECT-INDEPENDENT")
    print("="*80)
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    bagging_si_results = []
    method_start = time.time()

    for task_idx, (d1, d2, task_name) in enumerate(TASKS, 1):
        print(f"\nTASK {task_idx}/3: {task_name}")

        X1, y1, g1 = load_data_grouped(f"{BASE_DATA_PATH}/{d1}", 0)
        X2, y2, g2 = load_data_grouped(f"{BASE_DATA_PATH}/{d2}", 1)
        X = np.vstack((X1, X2))
        y = np.hstack((y1, y2))
        groups = np.hstack((g1, g2))
        del X1, X2, y1, y2, g1, g2
        gc.collect()

        gkf = GroupKFold(n_splits=5)
        fold_accs = []

        for fold_idx, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Bagging with reduced complexity
            bagging_clf = BaggingClassifier(
                estimator=AdaBoostClassifier(n_estimators=30, random_state=42),
                n_estimators=8,
                max_samples=0.7,
                max_features=0.8,
                random_state=42,
                n_jobs=2
            )

            bagging_clf.fit(X_train_scaled, y_train)
            y_pred = bagging_clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            fold_accs.append(acc)

            print(f"  Fold {fold_idx}: {acc:.4f}")

            del X_train_scaled, X_test_scaled, bagging_clf, y_pred
            gc.collect()

        avg_acc = np.mean(fold_accs)
        bagging_si_results.append({'Task': task_name, 'Accuracy': avg_acc, 'Std': np.std(fold_accs)})
        print(f"  Avg: {avg_acc:.4f}\n")

        del X, y, groups
        gc.collect()

    df = pd.DataFrame(bagging_si_results)
    df.to_csv(f"{ENSEMBLE_PATH}/Bagging_SubjectIndependent.csv", index=False)
    print(f"✅ BAGGING SI COMPLETE | Avg: {df['Accuracy'].mean():.4f} | Time: {(time.time()-method_start)/60:.1f}min\n")

except Exception as e:
    print(f"❌ ERROR in Bagging SI: {e}\n")

gc.collect()
print(f"🕐 Checkpoint: {datetime.now().strftime('%H:%M:%S')}")


METHOD 2: BAGGING ENSEMBLE - SUBJECT-INDEPENDENT
Started: 2026-01-30 13:20:54


TASK 1/3: Relax-Stroop
  Fold 1: 0.6720
  Fold 2: 0.6129
  Fold 3: 0.4763
  Fold 4: 0.5344
  Fold 5: 0.5752
  Avg: 0.5742


TASK 2/3: Relax-Arithmetic
  Fold 1: 0.5367
  Fold 2: 0.5446
  Fold 3: 0.4498
  Fold 4: 0.4798
  Fold 5: 0.5229
  Avg: 0.5068


TASK 3/3: Relax-Mirror
  Fold 1: 0.5670
  Fold 2: 0.5986
  Fold 3: 0.4497
  Fold 4: 0.4910
  Fold 5: 0.5399
  Avg: 0.5292

✅ BAGGING SI COMPLETE | Avg: 0.5367 | Time: 32.3min

🕐 Checkpoint: 13:53:13


In [ ]:
# ============================================================
# METHOD 1: VOTING - SUBJECT-DEPENDENT (OPTIMIZED)
# Memory-efficient, faster execution
# ============================================================

try:
    print("\n" + "="*80)
    print("METHOD 1: VOTING ENSEMBLE - SUBJECT-DEPENDENT")
    print("="*80)
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    voting_sd_results = []
    method_start = time.time()

    for task_idx, (d1, d2, task_name) in enumerate(TASKS, 1):
        print(f"\nTASK {task_idx}/3: {task_name}")

        X1, y1 = load_data_stratified(f"{BASE_DATA_PATH}/{d1}", 0)
        X2, y2 = load_data_stratified(f"{BASE_DATA_PATH}/{d2}", 1)
        X = np.vstack((X1, X2))
        y = np.hstack((y1, y2))
        del X1, X2, y1, y2
        gc.collect()

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_accs = []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Faster models (reduced estimators)
            base_models = [
                ('RF', RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=2, random_state=42)),
                ('XGB', xgb.XGBClassifier(n_estimators=50, max_depth=5, n_jobs=1, random_state=42, eval_metric='logloss')),
                ('ADA', AdaBoostClassifier(n_estimators=50, learning_rate=0.8, random_state=42)),
                ('CAT', CatBoostClassifier(iterations=50, depth=5, random_state=42, verbose=False, thread_count=2))
            ]

            voting_clf = VotingClassifier(estimators=base_models, voting='soft', n_jobs=1)
            voting_clf.fit(X_train_scaled, y_train)
            y_pred = voting_clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            fold_accs.append(acc)

            print(f"  Fold {fold_idx}: {acc:.4f}")

            # Cleanup each fold
            del X_train_scaled, X_test_scaled, voting_clf, y_pred
            gc.collect()

        avg_acc = np.mean(fold_accs)
        voting_sd_results.append({'Task': task_name, 'Accuracy': avg_acc, 'Std': np.std(fold_accs)})
        print(f"  Avg: {avg_acc:.4f}\n")

        del X, y
        gc.collect()

    df = pd.DataFrame(voting_sd_results)
    df.to_csv(f"{ENSEMBLE_PATH}/Voting_SubjectDependent.csv", index=False)
    print(f"✅ VOTING SD COMPLETE | Avg: {df['Accuracy'].mean():.4f} | Time: {(time.time()-method_start)/60:.1f}min\n")

except Exception as e:
    print(f"❌ ERROR in Voting SD: {e}\n")

# Force cleanup
gc.collect()
print(f"🕐 Checkpoint: {datetime.now().strftime('%H:%M:%S')}")


METHOD 1: VOTING ENSEMBLE - SUBJECT-DEPENDENT
Started: 2026-01-30 14:07:35


TASK 1/3: Relax-Stroop
  Fold 1: 0.7900
  Fold 2: 0.7901
  Fold 3: 0.7891
  Fold 4: 0.7881
  Fold 5: 0.7869
  Avg: 0.7888


TASK 2/3: Relax-Arithmetic
  Fold 1: 0.7582
  Fold 2: 0.7621
  Fold 3: 0.7593
  Fold 4: 0.7616
  Fold 5: 0.7596
  Avg: 0.7601


TASK 3/3: Relax-Mirror
  Fold 1: 0.7642
  Fold 2: 0.7660
  Fold 3: 0.7678
  Fold 4: 0.7662
  Fold 5: 0.7646
  Avg: 0.7658

✅ VOTING SD COMPLETE | Avg: 0.7716 | Time: 54.3min

🕐 Checkpoint: 15:01:54


In [ ]:
# ============================================================
# METHOD 2: BAGGING - SUBJECT-DEPENDENT (OPTIMIZED)
# ============================================================

try:
    print("\n" + "="*80)
    print("METHOD 2: BAGGING ENSEMBLE - SUBJECT-DEPENDENT")
    print("="*80)
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    bagging_sd_results = []
    method_start = time.time()

    for task_idx, (d1, d2, task_name) in enumerate(TASKS, 1):
        print(f"\nTASK {task_idx}/3: {task_name}")

        X1, y1 = load_data_stratified(f"{BASE_DATA_PATH}/{d1}", 0)
        X2, y2 = load_data_stratified(f"{BASE_DATA_PATH}/{d2}", 1)
        X = np.vstack((X1, X2))
        y = np.hstack((y1, y2))
        del X1, X2, y1, y2
        gc.collect()

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_accs = []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            bagging_clf = BaggingClassifier(
                estimator=AdaBoostClassifier(n_estimators=30, random_state=42),
                n_estimators=8,
                max_samples=0.7,
                max_features=0.8,
                random_state=42,
                n_jobs=2
            )

            bagging_clf.fit(X_train_scaled, y_train)
            y_pred = bagging_clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            fold_accs.append(acc)

            print(f"  Fold {fold_idx}: {acc:.4f}")

            del X_train_scaled, X_test_scaled, bagging_clf, y_pred
            gc.collect()

        avg_acc = np.mean(fold_accs)
        bagging_sd_results.append({'Task': task_name, 'Accuracy': avg_acc, 'Std': np.std(fold_accs)})
        print(f"  Avg: {avg_acc:.4f}\n")

        del X, y
        gc.collect()

    df = pd.DataFrame(bagging_sd_results)
    df.to_csv(f"{ENSEMBLE_PATH}/Bagging_SubjectDependent.csv", index=False)
    print(f"✅ BAGGING SD COMPLETE | Avg: {df['Accuracy'].mean():.4f} | Time: {(time.time()-method_start)/60:.1f}min\n")

except Exception as e:
    print(f"❌ ERROR in Bagging SD: {e}\n")

gc.collect()
print(f"🕐 Checkpoint: {datetime.now().strftime('%H:%M:%S')}")


METHOD 2: BAGGING ENSEMBLE - SUBJECT-DEPENDENT
Started: 2026-01-30 15:06:52


TASK 1/3: Relax-Stroop
  Fold 1: 0.6587
  Fold 2: 0.6576
  Fold 3: 0.6572
  Fold 4: 0.6568
  Fold 5: 0.6538
  Avg: 0.6568


TASK 2/3: Relax-Arithmetic
  Fold 1: 0.6109
  Fold 2: 0.6133
  Fold 3: 0.6120
  Fold 4: 0.6110
  Fold 5: 0.6080
  Avg: 0.6111


TASK 3/3: Relax-Mirror
  Fold 1: 0.6213
  Fold 2: 0.6310
  Fold 3: 0.6287
  Fold 4: 0.6300
  Fold 5: 0.6214
  Avg: 0.6265

✅ BAGGING SD COMPLETE | Avg: 0.6314 | Time: 32.3min

🕐 Checkpoint: 15:39:11


In [ ]:
# ============================================================
# METHOD 3: STACKING - SUBJECT-DEPENDENT (OPTIMIZED)
# ============================================================

try:
    print("\n" + "="*80)
    print("METHOD 3: STACKING ENSEMBLE - SUBJECT-DEPENDENT")
    print("="*80)
    print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

    stacking_sd_results = []
    method_start = time.time()

    for task_idx, (d1, d2, task_name) in enumerate(TASKS, 1):
        print(f"\nTASK {task_idx}/3: {task_name}")

        X1, y1 = load_data_stratified(f"{BASE_DATA_PATH}/{d1}", 0)
        X2, y2 = load_data_stratified(f"{BASE_DATA_PATH}/{d2}", 1)
        X = np.vstack((X1, X2))
        y = np.hstack((y1, y2))
        del X1, X2, y1, y2
        gc.collect()

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        fold_accs = []

        for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)

            # Reduced complexity base models
            base_models = {
                'RF': RandomForestClassifier(n_estimators=50, max_depth=15, n_jobs=2, random_state=42),
                'XGB': xgb.XGBClassifier(n_estimators=50, max_depth=5, n_jobs=1, random_state=42, eval_metric='logloss'),
                'ADA': AdaBoostClassifier(n_estimators=50, learning_rate=0.8, random_state=42)
            }

            # Train base models
            base_train_preds, base_test_preds = [], []
            for name, model in base_models.items():
                model.fit(X_train_scaled, y_train)
                if hasattr(model, 'predict_proba'):
                    base_train_preds.append(model.predict_proba(X_train_scaled)[:, 1])
                    base_test_preds.append(model.predict_proba(X_test_scaled)[:, 1])
                else:
                    base_train_preds.append(model.predict(X_train_scaled).astype(float))
                    base_test_preds.append(model.predict(X_test_scaled).astype(float))

            # Meta-learner
            meta_train = np.column_stack(base_train_preds)
            meta_test = np.column_stack(base_test_preds)
            meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
            meta_model.fit(meta_train, y_train)
            y_pred = meta_model.predict(meta_test)
            acc = accuracy_score(y_test, y_pred)
            fold_accs.append(acc)

            print(f"  Fold {fold_idx}: {acc:.4f}")

            del X_train_scaled, X_test_scaled, meta_train, meta_test, base_train_preds, base_test_preds
            gc.collect()

        avg_acc = np.mean(fold_accs)
        stacking_sd_results.append({'Task': task_name, 'Accuracy': avg_acc, 'Std': np.std(fold_accs)})
        print(f"  Avg: {avg_acc:.4f}\n")

        del X, y
        gc.collect()

    df = pd.DataFrame(stacking_sd_results)
    df.to_csv(f"{ENSEMBLE_PATH}/Stacking_SubjectDependent.csv", index=False)
    print(f"✅ STACKING SD COMPLETE | Avg: {df['Accuracy'].mean():.4f} | Time: {(time.time()-method_start)/60:.1f}min\n")

except Exception as e:
    print(f"❌ ERROR in Stacking SD: {e}\n")

gc.collect()
print(f"🕐 Checkpoint: {datetime.now().strftime('%H:%M:%S')}")


METHOD 3: STACKING ENSEMBLE - SUBJECT-DEPENDENT
Started: 2026-01-30 15:43:17


TASK 1/3: Relax-Stroop
  Fold 1: 0.8346
  Fold 2: 0.8367
  Fold 3: 0.8361
  Fold 4: 0.8357
  Fold 5: 0.8348
  Avg: 0.8356


TASK 2/3: Relax-Arithmetic
  Fold 1: 0.8005
  Fold 2: 0.8002
  Fold 3: 0.8019
  Fold 4: 0.8014
  Fold 5: 0.7992
  Avg: 0.8006


TASK 3/3: Relax-Mirror
  Fold 1: 0.7983
  Fold 2: 0.8007
  Fold 3: 0.8042
  Fold 4: 0.8028
  Fold 5: 0.8012
  Avg: 0.8014

✅ STACKING SD COMPLETE | Avg: 0.8126 | Time: 53.8min

🕐 Checkpoint: 16:37:08


In [ ]:
# ============================================================
# FINAL COMPREHENSIVE SUMMARY
# ============================================================

print("\n" + "="*80)
print("COMPLETE ENSEMBLE ANALYSIS SUMMARY")
print("="*80 + "\n")

# Load all results
results = {}
for method in ['Voting', 'Bagging', 'Stacking']:
    for val_type in ['SubjectIndependent', 'SubjectDependent']:
        file = f"{ENSEMBLE_PATH}/{method}_{val_type}.csv"
        if os.path.exists(file):
            df = pd.read_csv(file)
            results[f"{method}_{val_type}"] = df['Accuracy'].mean()

summary_df = pd.DataFrame(list(results.items()), columns=['Method', 'Accuracy'])
print(summary_df.to_string(index=False))
print(f"\n✅ ALL 15 EXPERIMENTS COMPLETE")
print(f"📁 Results: {ENSEMBLE_PATH}/")
print("="*80)



COMPLETE ENSEMBLE ANALYSIS SUMMARY

                    Method  Accuracy
 Voting_SubjectIndependent  0.504152
   Voting_SubjectDependent  0.771576
Bagging_SubjectIndependent  0.536715
  Bagging_SubjectDependent  0.631444
 Stacking_SubjectDependent  0.812553

✅ ALL 15 EXPERIMENTS COMPLETE
📁 Results: /path/to/Kaggle Dataset/Results/ResultsAfterOptimization/Complete_Ensemble_Analysis/
